# Compare / browse manifests

Notebook companion to `compare_manifests.py`. Browse manifest items and **click to play audio** — the player streams to your local machine over VS Code Remote-SSH, so you hear it on your laptop.

No `ipywidgets` needed. Navigate by calling `nxt()`, `prev()`, or `goto(n)` in a cell and re-running it.

Set the manifest path(s) below and run all cells.

In [ ]:
# --- Configure here -------------------------------------------------
MANIFESTS = [
    "/data-server/datasets/audio/nemo/diarization/en/ami/timestamps_asr/train.jsonl",
    # add a second manifest path here to compare them side by side
]

In [ ]:
import os
import soundfile as sf
from IPython.display import Audio, display

from compare_manifests import build_merged_data

merged_data = build_merged_data(MANIFESTS)
keys = list(merged_data.keys())
state = {"i": 0}
print(f"Loaded {len(keys)} items from {len(MANIFESTS)} manifest(s).")

In [ ]:
def _strip_common_path(paths):
    if len(paths) <= 1:
        return {p: os.path.basename(p) for p in paths}
    common = os.path.commonpath(paths)
    return {p: os.path.relpath(p, common) for p in paths}

def _extract_audio_turns(item):
    """Yield (path, offset, duration) for each 'audio' turn. offset/duration may be None."""
    turns = []
    for turn in item.get("conversations", []):
        if turn.get("type") == "audio" and turn.get("value"):
            turns.append((turn["value"], turn.get("offset"), turn.get("duration")))
    return turns

def _play_segment(path, offset, duration):
    """Display an audio player for the referenced [offset, offset+duration] slice."""
    info = sf.info(path)
    sr = info.samplerate
    start = int(round((offset or 0.0) * sr))
    frames = int(round(duration * sr)) if duration else -1
    data, sr = sf.read(path, start=start, frames=frames, dtype="float32")
    # IPython.display.Audio wants shape (channels, samples) for multi-channel
    if data.ndim > 1:
        data = data.T
    display(Audio(data, rate=sr))

def show(index=None):
    """Render the item at `index` (default: current position) with audio players."""
    if index is not None:
        state["i"] = index
    state["i"] = max(0, min(len(keys) - 1, state["i"]))
    i = state["i"]
    key = keys[i]
    items = merged_data[key]
    manifest_display = _strip_common_path(list(items.keys()))

    print(f"===== Item {i + 1} / {len(keys)}  —  id: {key} =====")

    max_len = max(len(it.get("conversations", [])) for it in items.values())
    for conv_idx in range(max_len):
        print(f"\n\U0001F4CB Conversation #{conv_idx}")
        for manifest, item in items.items():
            conv_list = item.get("conversations", [])
            if conv_idx >= len(conv_list):
                continue
            conv = conv_list[conv_idx]
            label = manifest_display[manifest] if len(items) > 1 else ""
            prefix = f"  [{label}] " if label else "  "
            for field in sorted(k for k in conv.keys() if k != "type"):
                val = str(conv[field])
                if "\n" in val:
                    print(f"{prefix}{field}:")
                    print('    """')
                    for line in val.split("\n"):
                        print(f"    {line}")
                    print('    """')
                else:
                    print(f"{prefix}{field}: {val}")

    seen = []
    for item in items.values():
        for path, offset, duration in _extract_audio_turns(item):
            if (path, offset, duration) in seen:
                continue
            seen.append((path, offset, duration))
            if not os.path.exists(path):
                print(f"\n⚠ File not found: {path}")
                continue
            seg = f" [{offset or 0:.3f}s → {(offset or 0) + duration:.3f}s]" if duration else " (full file)"
            print(f"\n▶ {path}{seg}")
            _play_segment(path, offset, duration)
    if not seen:
        print("\n⚠ No audio turns found in this item.")

def nxt():
    """Show the next item."""
    show(state["i"] + 1)

def prev():
    """Show the previous item."""
    show(state["i"] - 1)

def goto(n):
    """Jump to item number n (1-based)."""
    show(n - 1)

## Browse

Re-run the cell below to advance. Change it to `prev()` to go back, `goto(42)` to jump to item 42, or `show()` to redisplay the current one.

In [ ]:
nxt()